# 03 — Format Dataset YOLO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/03-object-detection/03_format_dataset_yolo.ipynb)

Untuk melatih YOLO dengan data sendiri, anotasi harus disusun dalam **format YOLO**. Notebook ini:
1. Menjelaskan struktur folder, file label `.txt`, dan `data.yaml`.
2. Membuat **dataset sintetis kecil** (gambar bentuk geometri) lengkap dengan labelnya, sepenuhnya dari kode.
3. Membaca kembali label dan memvisualisasikannya untuk verifikasi.
4. Mengonversi anotasi dari format **COCO** dan **Pascal VOC** ke YOLO.
5. Memvalidasi dataset, lalu membuktikan format sudah benar dengan melatih YOLO sebentar.

> ⚖️ **Lisensi:** `ultralytics` berlisensi **AGPL-3.0**.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**. Inference YOLO tetap bisa berjalan di CPU (model `n` cukup ringan), tetapi GPU jauh lebih cepat, terutama untuk training.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi

`ultralytics` belum terpasang di Colab secara default. Perhatikan lisensinya: **AGPL-3.0**. Bebas dipakai untuk belajar dan proyek open-source, tetapi jika dipakai di produk/layanan yang tidak dibuka kodenya, perlu lisensi Enterprise dari Ultralytics.

In [ ]:
%pip install -q ultralytics

## 2. Struktur Format YOLO

### 2.1 Struktur folder

```
dataset_bentuk/
├── data.yaml
├── images/
│   ├── train/   img_000.jpg, img_001.jpg, ...
│   └── val/     img_000.jpg, ...
└── labels/
    ├── train/   img_000.txt, img_001.txt, ...
    └── val/     img_000.txt, ...
```

Aturannya: setiap gambar `images/<split>/X.jpg` punya file label `labels/<split>/X.txt` dengan **nama yang sama**. YOLO menemukan label dengan mengganti `images` menjadi `labels` di path gambar.

### 2.2 File label `.txt`

Satu baris = satu objek, dengan 5 angka dipisahkan spasi:

```
<class_id> <x_center> <y_center> <width> <height>
```

- `class_id`: indeks kelas, **mulai dari 0**.
- Empat angka lainnya **ternormalisasi** (dibagi lebar/tinggi gambar), jadi bernilai 0–1. Karena itu label tetap valid walaupun gambar di-resize.
- Gambar tanpa objek boleh punya file `.txt` kosong (atau tanpa file); ini disebut *background image*.

Contoh gambar 640×480 dengan objek kelas 1 di kotak piksel (100, 50)–(300, 250):
`1 0.312500 0.312500 0.312500 0.416667`

### 2.3 File `data.yaml`

```yaml
path: /content/dataset_bentuk   # folder root dataset
train: images/train             # relatif terhadap path
val: images/val
names:
  0: lingkaran
  1: persegi
  2: segitiga
```

## 3. Membuat Dataset Sintetis

Agar notebook bisa langsung dijalankan tanpa mengunduh apa pun, kita membuat dataset sendiri: gambar 320×320 berisi 1–4 bentuk (**lingkaran**, **persegi**, **segitiga**) dengan warna, ukuran, dan posisi acak.

Karena kita sendiri yang menggambar bentuknya, kita tahu **persis** koordinat bounding box-nya, sehingga label dijamin benar. Bentuk boleh saling menutupi sebagian, seperti objek di foto nyata; kotak labelnya tetap mencakup seluruh bentuk.

In [ ]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
ROOT = Path("dataset_bentuk").resolve()   # path absolut dihitung saat runtime
CLASS_NAMES = ["lingkaran", "persegi", "segitiga"]
IMG_SIZE = 320
N_TRAIN, N_VAL = (40, 10) if QUICK_RUN else (200, 50)

random.seed(0)
np.random.seed(0)
print("QUICK_RUN:", QUICK_RUN, "| root:", ROOT)

Fungsi `buat_gambar` menggambar bentuk dan mengembalikan daftar objek dalam format piksel `xyxy`. Konversi ke format YOLO dilakukan terpisah di `xyxy_to_yolo_line`, dengan fungsi yang sama seperti di notebook 01.

In [ ]:
def buat_gambar(size=IMG_SIZE, max_obj=4):
    bg = tuple(int(v) for v in np.random.randint(180, 256, 3))
    img = Image.new("RGB", (size, size), bg)
    draw = ImageDraw.Draw(img)
    objek = []  # list (class_id, x1, y1, x2, y2)
    for _ in range(random.randint(1, max_obj)):
        cls = random.randrange(len(CLASS_NAMES))
        s = random.randint(30, 90)                       # ukuran bentuk
        x1, y1 = random.randint(0, size - s), random.randint(0, size - s)
        x2, y2 = x1 + s, y1 + s
        warna = tuple(int(v) for v in np.random.randint(0, 150, 3))
        if cls == 0:
            draw.ellipse([x1, y1, x2, y2], fill=warna)
        elif cls == 1:
            draw.rectangle([x1, y1, x2, y2], fill=warna)
        else:
            draw.polygon([((x1 + x2) / 2, y1), (x1, y2), (x2, y2)], fill=warna)
        objek.append((cls, x1, y1, x2, y2))
    return img, objek


def xyxy_to_yolo_line(cls, x1, y1, x2, y2, w, h):
    cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
    bw, bh = (x2 - x1) / w, (y2 - y1) / h
    return f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


if ROOT.exists():
    shutil.rmtree(ROOT)  # mulai dari folder bersih

for split, n in [("train", N_TRAIN), ("val", N_VAL)]:
    (ROOT / "images" / split).mkdir(parents=True)
    (ROOT / "labels" / split).mkdir(parents=True)
    for i in range(n):
        img, objek = buat_gambar()
        img.save(ROOT / "images" / split / f"img_{i:03d}.jpg", quality=95)
        baris = [xyxy_to_yolo_line(*o, IMG_SIZE, IMG_SIZE) for o in objek]
        (ROOT / "labels" / split / f"img_{i:03d}.txt").write_text("\n".join(baris) + "\n")

print(f"Dibuat {N_TRAIN} gambar train dan {N_VAL} gambar val")

### 3.1 Menulis `data.yaml`

Kita tulis dengan library `yaml` (PyYAML, sudah terpasang bersama ultralytics). `path` diisi path absolut hasil `Path.resolve()` di atas, jadi tidak ada path yang di-*hardcode*.

In [ ]:
import yaml

data_cfg = {
    "path": str(ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
DATA_YAML = ROOT / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True))
print(DATA_YAML.read_text())

### 3.2 Melihat isi file

Mari lihat struktur folder dan isi satu file label.

In [ ]:
for split in ["train", "val"]:
    n_img = len(list((ROOT / "images" / split).glob("*.jpg")))
    n_lbl = len(list((ROOT / "labels" / split).glob("*.txt")))
    print(f"{split:<5}: {n_img} gambar, {n_lbl} label")

contoh_label = ROOT / "labels" / "train" / "img_000.txt"
print(f"\nIsi {contoh_label.name}:")
print(contoh_label.read_text())

## 4. Membaca Label dan Memvisualisasikannya

**Selalu verifikasi label secara visual** sebelum training. Label yang salah (misalnya koordinat tertukar atau lupa dinormalisasi) adalah penyebab paling umum model yang "tidak mau belajar".

Langkah membaca: `cx, cy, w, h` ternormalisasi → kalikan ukuran gambar → ubah ke `xyxy` piksel → gambar.

In [ ]:
def baca_label(path_label, img_w, img_h):
    objek = []
    for baris in Path(path_label).read_text().strip().splitlines():
        cls, cx, cy, w, h = baris.split()
        cx, cy, w, h = float(cx) * img_w, float(cy) * img_h, float(w) * img_w, float(h) * img_h
        objek.append((int(cls), cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return objek


def tampilkan_dengan_label(path_img, ax):
    img = Image.open(path_img)
    path_label = Path(str(path_img).replace("/images/", "/labels/")).with_suffix(".txt")
    ax.imshow(img)
    warna = ["tab:red", "tab:blue", "tab:green"]
    for cls, x1, y1, x2, y2 in baca_label(path_label, *img.size):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=warna[cls], lw=2))
        ax.text(x1, y1, CLASS_NAMES[cls], color="white", fontsize=8, va="bottom",
                bbox=dict(facecolor=warna[cls], pad=1, edgecolor="none"))
    ax.set_title(Path(path_img).name, fontsize=9)
    ax.axis("off")


fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, p in zip(axes.flat, sorted((ROOT / "images" / "train").glob("*.jpg"))[:8]):
    tampilkan_dengan_label(p, ax)
plt.tight_layout()
plt.show()

## 5. Konversi dari Format Lain

Dataset publik sering tersedia dalam format lain. Dua yang paling umum:

| Format | Penyimpanan | Kotak | Kelas |
|---|---|---|---|
| **COCO** | satu file JSON untuk semua gambar | `[x_min, y_min, w, h]` piksel | `category_id` (sering mulai dari 1, tidak berurutan) |
| **Pascal VOC** | satu file XML per gambar | `xmin, ymin, xmax, ymax` piksel | nama kelas (string) |

### 5.1 COCO → YOLO

Hal yang perlu diperhatikan: `category_id` COCO harus dipetakan ulang ke indeks yang dimulai dari 0 dan berurutan.

In [ ]:
coco = {
    "images": [{"id": 7, "file_name": "foto.jpg", "width": 640, "height": 480}],
    "categories": [{"id": 3, "name": "mobil"}, {"id": 18, "name": "anjing"}],
    "annotations": [
        {"image_id": 7, "category_id": 3,  "bbox": [100, 50, 200, 200]},
        {"image_id": 7, "category_id": 18, "bbox": [400, 300, 120, 90]},
    ],
}

def coco_to_yolo(coco):
    # Petakan category_id COCO (3, 18, ...) ke indeks YOLO berurutan (0, 1, ...)
    peta_kelas = {c["id"]: i for i, c in enumerate(coco["categories"])}
    info = {im["id"]: im for im in coco["images"]}
    hasil = {}
    for a in coco["annotations"]:
        im = info[a["image_id"]]
        x, y, w, h = a["bbox"]
        baris = xyxy_to_yolo_line(peta_kelas[a["category_id"]], x, y, x + w, y + h, im["width"], im["height"])
        hasil.setdefault(Path(im["file_name"]).with_suffix(".txt").name, []).append(baris)
    return hasil, [c["name"] for c in coco["categories"]]

label_yolo, nama_kelas = coco_to_yolo(coco)
print("names:", nama_kelas)
for f, baris in label_yolo.items():
    print(f"{f}:"); print("\n".join(baris))

### 5.2 Pascal VOC → YOLO

VOC memakai XML. Kita parse dengan `xml.etree.ElementTree` dari standard library Python.

In [ ]:
import xml.etree.ElementTree as ET

voc_xml = """
<annotation>
  <filename>jalan.jpg</filename>
  <size><width>800</width><height>600</height><depth>3</depth></size>
  <object><name>orang</name><bndbox><xmin>120</xmin><ymin>200</ymin><xmax>220</xmax><ymax>520</ymax></bndbox></object>
  <object><name>sepeda</name><bndbox><xmin>300</xmin><ymin>350</ymin><xmax>520</xmax><ymax>560</ymax></bndbox></object>
</annotation>
"""

def voc_to_yolo(xml_text, class_names):
    root = ET.fromstring(xml_text)
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)
    baris = []
    for obj in root.iter("object"):
        cls = class_names.index(obj.find("name").text)
        bb = obj.find("bndbox")
        x1, y1, x2, y2 = (float(bb.find(k).text) for k in ["xmin", "ymin", "xmax", "ymax"])
        baris.append(xyxy_to_yolo_line(cls, x1, y1, x2, y2, w, h))
    return baris

print("\n".join(voc_to_yolo(voc_xml, ["orang", "sepeda"])))

## 6. Validasi Dataset

Sebelum training, jalankan pemeriksaan otomatis untuk kesalahan umum:
- Setiap gambar punya file label (atau memang sengaja tanpa objek).
- Setiap baris berisi tepat 5 nilai.
- `class_id` berada di rentang `0 … jumlah_kelas − 1`.
- Koordinat berada di rentang `[0, 1]`, dan lebar/tinggi > 0.

In [ ]:
def validasi_dataset(root, n_kelas):
    masalah, jumlah_objek = [], np.zeros(n_kelas, dtype=int)
    for split in ["train", "val"]:
        for img in sorted((root / "images" / split).glob("*.jpg")):
            lbl = root / "labels" / split / (img.stem + ".txt")
            if not lbl.exists():
                masalah.append(f"{img.name}: tidak ada file label")
                continue
            for no, baris in enumerate(lbl.read_text().strip().splitlines(), 1):
                nilai = baris.split()
                if len(nilai) != 5:
                    masalah.append(f"{lbl.name}:{no} harus 5 nilai, ada {len(nilai)}")
                    continue
                cls, *koordinat = int(nilai[0]), *map(float, nilai[1:])
                if not 0 <= cls < n_kelas:
                    masalah.append(f"{lbl.name}:{no} class_id {cls} di luar rentang")
                if not all(0 <= v <= 1 for v in koordinat) or koordinat[2] <= 0 or koordinat[3] <= 0:
                    masalah.append(f"{lbl.name}:{no} koordinat tidak valid {koordinat}")
                if 0 <= cls < n_kelas:
                    jumlah_objek[cls] += 1
    return masalah, jumlah_objek


masalah, jumlah_objek = validasi_dataset(ROOT, len(CLASS_NAMES))
print("Masalah ditemukan:", len(masalah))
for m in masalah[:10]:
    print(" -", m)
print("\nDistribusi objek per kelas:")
for n, j in zip(CLASS_NAMES, jumlah_objek):
    print(f"  {n:<10} {j:>4}  {'█' * (j // 5)}")

## 7. Bukti Format Benar: Training Singkat

Cara paling meyakinkan untuk mengecek format adalah melatih YOLO dengan dataset ini. Kita *fine-tune* `yolo11n.pt` (pretrained COCO) ke 3 kelas bentuk.

- `data`: path ke `data.yaml`.
- `epochs`: 30 di GPU sudah cukup untuk dataset sederhana ini (`QUICK_RUN` hanya 1 epoch, jadi hasilnya belum bermakna).
- `imgsz=320`: sesuai ukuran gambar kita.

Output training disimpan di folder `runs/detect/…`. Perhatikan kolom `mAP50` dan `mAP50-95`, metrik yang kita bangun sendiri di notebook 01.

In [ ]:
from ultralytics import YOLO

EPOCHS = 1 if QUICK_RUN else 30
model = YOLO("yolo11n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=16,
    workers=2,
    plots=not QUICK_RUN,
    verbose=False,
)

### 7.1 Evaluasi dan prediksi

`model.val()` menghitung metrik di data validasi. Setelah itu, kita tampilkan prediksi di beberapa gambar val, berdampingan dengan label aslinya.

In [ ]:
metrics = model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, verbose=False, plots=False)
print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")

val_imgs = sorted((ROOT / "images" / "val").glob("*.jpg"))[:4]
preds = model.predict(val_imgs, imgsz=IMG_SIZE, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for i, (p, r) in enumerate(zip(val_imgs, preds)):
    tampilkan_dengan_label(p, axes[0, i])
    axes[0, i].set_title(f"label asli: {p.name}", fontsize=9)
    axes[1, i].imshow(r.plot()[..., ::-1])
    axes[1, i].set_title(f"prediksi: {len(r.boxes)} objek", fontsize=9)
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## Ringkasan

- Format YOLO: `images/<split>/X.jpg` ↔ `labels/<split>/X.txt`, satu baris per objek: `class cx cy w h` ternormalisasi 0–1.
- `data.yaml` berisi root dataset, folder train/val, dan nama kelas (indeks mulai 0).
- COCO (`xywh` piksel, JSON) dan VOC (`xyxy` piksel, XML) perlu dikonversi: ubah ke pusat, normalisasi, dan petakan ulang kelas.
- Selalu **visualisasikan dan validasi** label sebelum training.

## Latihan

1. **Temukan bug.** Buat salinan dataset lalu sengaja rusak 5 label dengan kesalahan umum: (a) koordinat tidak dinormalisasi (masih piksel), (b) memakai `x_min, y_min` alih-alih pusat, (c) `class_id` = 3, (d) lebar/tinggi tertukar, (e) file label dihapus. Jalankan `validasi_dataset` dan visualisasi. Kesalahan mana yang **tidak** tertangkap validator, dan bagaimana cara mendeteksinya?
2. **Tambah kelas & split test.** Tambahkan kelas keempat `bintang` (gambar dengan `draw.polygon` 10 titik) dan split `test`. Perbarui `data.yaml` (tambahkan `test: images/test`), latih ulang, lalu evaluasi di split test dengan `model.val(split="test")`. Kelas mana yang paling sulit, dan mengapa?
3. **Dataset nyata kecil.** Pilih 15–20 foto dari ponselmu dengan satu jenis objek (misalnya gelas atau botol). Anotasi dengan tool gratis seperti [Label Studio](https://labelstud.io) atau [CVAT](https://www.cvat.ai), ekspor ke format YOLO, susun folder dan `data.yaml` seperti di notebook ini, lalu latih `yolo11n.pt` di Colab. Tampilkan hasil prediksi di 3 foto yang tidak ikut training.